## The container threat model

Before stacking defences, be clear about what a container *is*. **A container is a process-isolation mechanism, not a security boundary.** It keeps workloads from tripping over each other on a shared kernel; it is **not** designed to safely host hostile, untrusted code on a multi-tenant kernel — that's what VMs (and sandboxes like gVisor, Kata Containers) are for. Every technique in this module assumes that framing.

**What container isolation protects against by default** (the module-01 primitives doing their job):

- one container reading or altering another's files (separate **mount** namespaces),
- one seeing another's processes (separate **PID** namespaces),
- one hogging all CPU/RAM (**cgroups** cap resources per container),
- one binding the host's privileged ports (**network** namespace).

**What it does *not* protect against by default:**

- a container running as **root** escaping via a kernel CVE (several land every year),
- a **`--privileged`** container doing essentially anything to the host,
- a container with `CAP_SYS_ADMIN` mounting host filesystems.

The kernel primitives are the toolbox: **namespaces** isolate views, **cgroups** limit resources, **capabilities** slice root into ~40 fine-grained privileges, **seccomp** allow/block-lists syscalls, **AppArmor/SELinux** add mandatory access control, and **user namespaces** remap container UIDs to unprivileged host UIDs. Docker's *default* config is already far tighter than running the process bare on the host — it drops ~26 of 40 capabilities and blocks ~50 syscalls out of the box. The rest of this module is about turning the remaining dials from "default" toward "least privilege," one primitive per section.